# Fine-tuning Models on SageMaker

This notebook demonstrates how to fine-tune a pre-trained model on Amazon SageMaker using the Amazon Reviews Polarity dataset.

In [ ]:
# Install required packages
!pip install -q "torch==1.13.1" "transformers==4.26.0" "datasets==2.10.1" "boto3>=1.26.0" "sagemaker>=2.130.0" "pandas>=1.5.0" "numpy>=1.23.0" "matplotlib>=3.6.0" "scikit-learn>=1.2.0"

In [ ]:
# Import required libraries
import os
import json
import time
import tarfile
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from datasets import Dataset
import boto3
import sagemaker
from sagemaker.huggingface import HuggingFace
from sagemaker.huggingface.model import HuggingFaceModel

# Set up SageMaker session
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.session.Session().region_name
bucket = sagemaker_session.default_bucket()
prefix = "fine-tuning-workshop"

# Set model parameters
base_model = "distilbert-base-uncased-finetuned-sst-2-english"
task = "sequence-classification"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForSequenceClassification.from_pretrained(base_model)

## 1. Download and Process the Amazon Polarity Dataset

The Amazon Polarity dataset has a simple format: each line contains a label (1 or 2), a title, and a review text. To avoid memory issues, we'll only load a subset of the data.

In [ ]:
# Create data directory
os.makedirs("data", exist_ok=True)

# Download the dataset
archive_url = "https://s3.amazonaws.com/fast-ai-nlp/amazon_review_polarity_csv.tgz"
archive_path = "data/amazon_review_polarity_csv.tgz"
extract_dir = "data/amazon_polarity_extracted"

# Download if not already present
if not os.path.exists(archive_path):
    print(f"Downloading from {archive_url}...")
    response = requests.get(archive_url, stream=True)
    with open(archive_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print(f"Download complete. File size: {os.path.getsize(archive_path) / (1024 * 1024):.2f} MB")
else:
    print(f"Archive already exists at {archive_path}")

# Extract the archive
os.makedirs(extract_dir, exist_ok=True)
print(f"Extracting archive to {extract_dir}...")
with tarfile.open(archive_path, "r:gz") as tar:
    tar.extractall(path=extract_dir)

In [ ]:
# Find the extracted files
all_files = []
for root, dirs, files in os.walk(extract_dir):
    for file in files:
        all_files.append(os.path.join(root, file))

print(f"Found {len(all_files)} files")

# Identify train file
train_file = None

for file in all_files:
    if "train" in file.lower():
        train_file = file
        break

# If not found by name, use the largest file
if not train_file:
    all_files.sort(key=os.path.getsize, reverse=True)
    if len(all_files) >= 1:
        train_file = all_files[0]

print(f"Using {train_file} as data file")

In [ ]:
# Load only the first 11,000 rows to avoid memory issues
print("Loading 11,000 rows from the dataset...")
all_data = pd.read_csv(train_file, header=None, names=['label', 'title', 'review'], nrows=11000)

# Convert labels from 1/2 to 0/1 for binary classification
all_data['label'] = all_data['label'] - 1

# Split into train (10,000) and test (1,000) sets
train_df = all_data[:10000]
test_df = all_data[10000:]

print(f"Train dataset: {len(train_df)} rows")
print(f"Test dataset: {len(test_df)} rows")

# Display a sample
print("\nSample from train dataset:")
print(train_df.head(3))

In [ ]:
# Convert to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

print(f"Final train set size: {len(train_dataset)}")
print(f"Final test set size: {len(test_dataset)}")

## 2. Prepare the Dataset for Fine-tuning

Now, let's tokenize the dataset and prepare it for fine-tuning.

In [ ]:
# Prepare the dataset for the model
def tokenize_function(examples):
    # Combine title and review for each example
    texts = [f"{title}. {review}" for title, review in zip(examples["title"], examples["review"])]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=128)

# Tokenize the datasets
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Set the format for PyTorch
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("Datasets prepared for fine-tuning")

## 3. Save Datasets to S3

Now, let's save our prepared datasets to S3 so they can be accessed by the SageMaker training job.

In [ ]:
# Create directories for datasets
os.makedirs("data/train", exist_ok=True)
os.makedirs("data/test", exist_ok=True)

# Save datasets locally
tokenized_train.save_to_disk("data/train")
tokenized_test.save_to_disk("data/test")

# Upload datasets to S3
train_s3_uri = sagemaker_session.upload_data(path="data/train", bucket=bucket, key_prefix=f"{prefix}/data/train")
test_s3_uri = sagemaker_session.upload_data(path="data/test", bucket=bucket, key_prefix=f"{prefix}/data/test")

print(f"Train dataset uploaded to: {train_s3_uri}")
print(f"Test dataset uploaded to: {test_s3_uri}")

## 4. Configure and Launch Distributed Fine-tuning Job

Now, let's configure and launch a distributed fine-tuning job on SageMaker using GPU instances.

**Note:** We've updated the training script (`train_sentiment.py`) to ensure compatibility between numpy and pandas versions in the SageMaker environment. The script now installs the required versions of numpy and pandas before importing them.

In [ ]:
# Define hyperparameters
hyperparameters = {
    "epochs": 3,
    "train-batch-size": 32,
    "eval-batch-size": 64,
    "learning-rate": 5e-5,
    "warmup-steps": 500,
    "model-id": base_model,
    "fp16": True,
    # Add gradient accumulation for better memory efficiency
    "gradient-accumulation-steps": 2,
    # Add error recovery options
    "auto-recover": True
}

# Define metric definitions for CloudWatch monitoring
metric_definitions = [
    {"Name": "train_loss", "Regex": "train_loss: ([0-9\\.]+)"},
    {"Name": "eval_loss", "Regex": "eval_loss: ([0-9\\.]+)"},
    {"Name": "eval_accuracy", "Regex": "eval_accuracy: ([0-9\\.]+)"},
    {"Name": "eval_f1", "Regex": "eval_f1: ([0-9\\.]+)"},
    {"Name": "eval_precision", "Regex": "eval_precision: ([0-9\\.]+)"},
    {"Name": "eval_recall", "Regex": "eval_recall: ([0-9\\.]+)"}
]

In [ ]:
# Configure the Hugging Face estimator for distributed training
huggingface_estimator = HuggingFace(
    entry_point="train_sentiment.py",
    source_dir="scripts",
    role=role,
    instance_count=2,  # Use 2 instances for distributed training
    instance_type="ml.g4dn.xlarge",  # GPU instance type
    transformers_version="4.26.0",
    pytorch_version="1.13.1",
    py_version="py39",
    hyperparameters=hyperparameters,
    metric_definitions=metric_definitions,
    distribution={
        "pytorchddp": {
            "enabled": True,  # Enable PyTorch Distributed Data Parallel
            "custom_mpi_options": "-verbose --mca orte_base_help_aggregate 0"  # Add verbose logging for debugging
        }
    },
    max_run=7200,  # Maximum runtime in seconds (2 hours)
    output_path=f"s3://{bucket}/{prefix}/output",
    # Add debugging configuration
    debugger_hook_config=False,  # Disable SageMaker Debugger for PyTorch DDP compatibility
    # Add retry configuration
    max_retry_attempts=2  # Retry failed jobs up to 2 times
)

# Define the data channels
data_channels = {
    "train": train_s3_uri,
    "test": test_s3_uri
}

In [ ]:
# Launch the training job
job_name = f"fine-tuning-{time.strftime('%Y-%m-%d-%H-%M-%S')}"
print(f"Launching training job: {job_name}")

huggingface_estimator.fit(data_channels, job_name=job_name)

print(f"Training job completed: {job_name}")

## 5. Deploy the Fine-tuned Model

Now that we have fine-tuned our model, let's deploy it to a SageMaker endpoint for inference.

In [ ]:
# Create a Hugging Face model from the trained artifacts
fine_tuned_model = HuggingFaceModel(
    model_data=huggingface_estimator.model_data,
    role=role,
    transformers_version="4.26.0",
    pytorch_version="1.13.1",
    py_version="py39"
)

# Deploy the model to an endpoint
endpoint_name = f"fine-tuned-sentiment-{time.strftime('%Y-%m-%d-%H-%M-%S')}"
print(f"Deploying model to endpoint: {endpoint_name}")

predictor = fine_tuned_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g4dn.xlarge",  # GPU instance for inference
    endpoint_name=endpoint_name
)

print(f"Model deployed to endpoint: {endpoint_name}")

## 6. Test the Deployed Model

Let's test our deployed model with some sample reviews.

In [ ]:
# Sample reviews for testing
sample_reviews = [
    "This product exceeded my expectations. The quality is excellent and it works perfectly.",
    "I'm very disappointed with this purchase. It broke after just one week of use.",
    "Average product, nothing special but it gets the job done.",
    "The customer service was excellent when I had an issue with my order.",
    "Would not recommend. Poor quality and overpriced for what you get."
]

# Send requests to the endpoint
for i, review in enumerate(sample_reviews):
    response = predictor.predict({
        "inputs": review
    })
    
    # Extract the prediction
    label = response[0]['label']
    score = response[0]['score']
    sentiment = "Positive" if label == "LABEL_1" else "Negative"
    
    print(f"Review {i+1}: {review}")
    print(f"Prediction: {sentiment} (confidence: {score:.4f})\n")

## 7. Clean Up Resources

To avoid incurring unnecessary charges, let's clean up the resources we created.

In [ ]:
# Delete the endpoint
print(f"Deleting endpoint: {endpoint_name}")
sagemaker_session.delete_endpoint(endpoint_name)
print("Endpoint deleted successfully.")